# Install kaggle-environments

In [1]:
# 1. Enable Internet in the Kernel (Settings side pane)

# 2. Curl cache may need purged if v0.1.6 cannot be found (uncomment if needed). 
# !curl -X PURGE https://pypi.org/simple/kaggle-environments

# ConnectX environment was defined in v0.1.6
!pip install 'kaggle-environments>=0.1.6'

  Using cached kaggle_environments-1.29.1-py3-none-any.whl.metadata (829 bytes)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached gymnasium-1.2.0-py3-none-any.whl.metadata (9.9 kB)
  Using cached gymnax-0.0.8-py3-none-any.whl.metadata (19 kB)
  Using cached jax-0.10.0-py3-none-any.whl.metadata (13 kB)
  Using cached litellm-1.82.4-py3-none-any.whl.metadata (30 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pettingzoo-1.24.0-py3-none-any.whl.metadata (8.1 kB)
  Using cached pokerkit-0.6.3-py3-none-any.whl.metadata (18 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pygame-2.6.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached pyjson5-2.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Usi

# Create ConnectX Environment

In [18]:
from kaggle_environments import evaluate, make, utils

env = make("connectx", debug=True)
env.render()

+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+



# Create an Agent

To create the submission, an agent function should be fully encapsulated (no external dependencies).  

When your agent is being evaluated against others, it will not have access to the Kaggle docker image.  Only the following can be imported: Python Standard Library Modules, gym, numpy, scipy, pytorch (1.3.1, cpu only), and more may be added later.



In [19]:
import random
from kaggle_environments import make

# Import ngắn gọn các "vũ khí" từ các file riêng biệt
from board import ConnectXBoard
from evaluator import BoardEvaluator
from ai import MinimaxAI

# =====================================================================
# KHỞI TẠO ĐỐI TƯỢNG TOÀN CỤC (GLOBAL STATE)
# =====================================================================
_EVALUATOR = None
_BOT_AI = None

# =====================================================================
# HÀM AGENT CHUẨN KAGGLE
# =====================================================================
def agent(observation, configuration):
    global _EVALUATOR, _BOT_AI
    
    rows = configuration.rows
    cols = configuration.columns
    x = configuration.inarow
    my_id = observation.mark - 1 # Đổi (1, 2) thành (0, 1)

    # Khởi tạo bộ não AI một lần duy nhất cho cả trận
    if _EVALUATOR is None:
        _EVALUATOR = BoardEvaluator(win_condition=x)
        _BOT_AI = MinimaxAI(max_depth=3, evaluator=_EVALUATOR)

    # 1. Tạo bàn cờ Bitboard rỗng
    game_board = ConnectXBoard(w=cols, h=rows, x=x)
    
    # 2. ĐỒNG BỘ: Đọc dữ liệu mảng phẳng từ Kaggle nạp vào Bitboard
    kaggle_board = observation.board
    for r in range(rows):
        for c in range(cols):
            val = kaggle_board[r * cols + c]
            if val != 0:
                p_id = val - 1
                row_from_bottom = rows - 1 - r
                bit_idx = c * game_board.col_height + row_from_bottom
                game_board.boards[p_id] |= (1 << bit_idx)
                
    # 3. Cập nhật lại mảng heights để sẵn sàng cho nước đi tiếp theo
    for c in range(cols):
        r_idx = 0
        while r_idx < rows and (game_board.boards[0] | game_board.boards[1]) & (1 << (c * game_board.col_height + r_idx)):
            r_idx += 1
        game_board.heights[c] = c * game_board.col_height + r_idx

    # 4. Gọi AI tính toán nước đi tốt nhất và trả về
    return int(_BOT_AI.choose_best_move(game_board, player_id=my_id))


# =====================================================================
# CHẠY THỬ NGHIỆM TRÊN KAGGLE ENVIRONMENT (LOCAL)
# =====================================================================
if __name__ == "__main__":
    # Tạo môi trường giả lập game ConnectX (mặc định là 7x6, bộ 4 quân)
    env = make("connectx", debug=True)
    
    # Cho Bot của mình đấu thử với đối thủ đi "Random"
    env.run([agent, "random"])
    
    # In kết quả trận đấu ra Terminal dưới dạng Text để kiểm tra
    print(env.render(mode="ansi"))

+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 1 | 0 | 0 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 1 | 0 | 1 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 1 | 0 | 2 | 2 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 1 | 0 | 2 | 1 | 2 | 0 | 2 |
+---+---+---+---+---+---+---+



# Test your Agent

In [20]:
env.reset()
# Play as the first agent against default "random" agent.
env.run([my_agent, "random"])
env.render(mode="ipython", width=500, height=450)

# Debug/Train your Agent

In [12]:
# Play as first position against random agent.
trainer = env.train([None, "random"])

observation = trainer.reset()

while not env.done:
    my_action = my_agent(observation, env.configuration)
    print("My Action", my_action)
    observation, reward, done, info = trainer.step(my_action)
    # env.render(mode="ipython", width=100, height=90, header=False, controls=False)
env.render()

My Action 0
My Action 4
My Action 0
My Action 3
My Action 5
My Action 3
My Action 3
My Action 5
My Action 6
My Action 2
My Action 5
My Action 4
My Action 5
My Action 3
My Action 1
My Action 1
My Action 1
My Action 2
My Action 4
My Action 6
My Action 6
+---+---+---+---+---+---+---+
| 2 | 1 | 1 | 1 | 2 | 2 | 2 |
+---+---+---+---+---+---+---+
| 2 | 1 | 2 | 2 | 1 | 1 | 1 |
+---+---+---+---+---+---+---+
| 1 | 2 | 2 | 2 | 1 | 2 | 2 |
+---+---+---+---+---+---+---+
| 2 | 1 | 1 | 1 | 2 | 1 | 1 |
+---+---+---+---+---+---+---+
| 2 | 2 | 2 | 1 | 2 | 1 | 1 |
+---+---+---+---+---+---+---+
| 1 | 2 | 2 | 1 | 1 | 1 | 2 |
+---+---+---+---+---+---+---+



# Evaluate your Agent

In [13]:
def mean_reward(rewards):
    return sum(r[0] for r in rewards) / float(len(rewards))

# Run multiple episodes to estimate its performance.
print("My Agent vs Random Agent:", mean_reward(evaluate("connectx", [my_agent, "random"], num_episodes=10)))
print("My Agent vs Negamax Agent:", mean_reward(evaluate("connectx", [my_agent, "negamax"], num_episodes=10)))

My Agent vs Random Agent: 1.0
My Agent vs Negamax Agent: 0.5


# Play your Agent
Click on any column to place a checker there ("manually select action").

In [14]:
# "None" represents which agent you'll manually play as (first or second player).
env.play([None, "negamax"], width=500, height=450)

AttributeError: 'Environment' object has no attribute 'play'

# Write Submission File



In [8]:
import inspect
import os

def write_agent_to_file(function, file):
    with open(file, "a" if os.path.exists(file) else "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

write_agent_to_file(my_agent, "submission.py")

<function my_agent at 0x7f53c74c68c8> written to submission.py


# Validate Submission
Play your submission against itself.  This is the first episode the competition will run to weed out erroneous agents.

Why validate? This roughly verifies that your submission is fully encapsulated and can be run remotely.

In [9]:
# Note: Stdout replacement is a temporary workaround.
import sys
out = sys.stdout
submission = utils.read_file("/kaggle/working/submission.py")
agent = utils.get_last_callable(submission)
sys.stdout = out

env = make("connectx", debug=True)
env.run([agent, agent])
print("Success!" if env.state[0].status == env.state[1].status == "DONE" else "Failed...")


Success!


# Submit to Competition

1. Commit this kernel.
2. View the commited version.
3. Go to "Data" section and find submission.py file.
4. Click "Submit to Competition"
5. Go to [My Submissions](https://kaggle.com/c/connectx/submissions) to view your score and episodes being played.